In [2]:
##modules
#%matplotlib widget
#%matplotlib inline
#
%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf
import numpy as np

import pandas as pd


import pickle

In [4]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_SELF import get_paths_SELF

# Parámetros editables
disco = "g"
layer_script = "event"
subj = "s01b"


# Generar variables automáticamente
path_dict = get_paths_SELF(disco=disco, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")


✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\ICA_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_matlab_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\evoked_event
✅ Carpeta creada: g:\PROYECTO_SELF\channels_structure
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\raw_hsp
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\fwd
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\inverse
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\acw_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\PLE_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\ISC_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\anal

In [5]:

layer_script= "event"  # Cambia esto si es necesario
pickle_file = epochs_clean_path / f"dict_conditions.pkl"
# Cargar el pickle
with open(pickle_file, "rb") as f:
    dict_conditions = pickle.load(f)

#epochs
combinaciones = list(dict_conditions.keys())
print(f"combinaciones: {combinaciones}")

combinacion= combinaciones[0]  # Selecciona la primera combinación

subj = sorted({f.name.split("_")[0].lower() for f in data_task_edf.glob("*.edf")})
print(f"subj: {subj}")


combinaciones: ['self_pos', 'self_neu', 'self_neg', 'friend_pos', 'friend_neu', 'friend_neg', 'unk_pos', 'unk_neu', 'unk_neg']
subj: ['s01b', 's02b', 's03b', 's04b', 's05b', 's06b', 's07b', 's08b', 's09b', 's10b', 's11b', 's12b', 's13b', 's14b', 's15b', 's16b', 's17b', 's18b', 's19b', 's20b', 's21b', 's22b', 's23b', 's24b', 's25b', 's26b', 's27b', 's28b', 's29b']


# LECTURA EPOCHS

In [6]:
epochs=mne.read_epochs(epochs_clean_path / f"{subj[0]}_epochs_{combinacion}_{layer_script}-epo.fif")

Reading g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s01b_epochs_self_pos_event-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    9000.00 ms
        0 CTF compensation matrices available
Not setting metadata
32 matching events found
No baseline correction applied
0 projection items activated


In [7]:

#se coge este sujeto, entiendo porque tiene los 272 canales comunes a todos los sujetos
epochs_eeg_clean = epochs.copy().pick(picks="eeg", exclude='bads')

print(len(epochs_eeg_clean.ch_names))  # Esto debería darte 272

# Canales efectivos reales del objeto epochs (ya vienen con -4304)
canales_efectivos = set(epochs_eeg_clean.ch_names)  # conjunto para acceso rápido



# Obtenemos matriz de adjacencia general, y el total de los canales
adjacency, ch_names_total = mne.channels.find_ch_adjacency(epochs.info, ch_type='eeg')
print(adjacency.shape)  # Esto debería darte (272, 272)

# Convertimos a string normal por si acaso (np.str_ a str)
ch_names_total_str = [str(ch) for ch in ch_names_total]

# Creamos la tabla de los canales con sus nombres y sufijos
df_canales = pd.DataFrame({
    'indice': range(len(ch_names_total_str)),
    'nombre_base': ch_names_total_str,
    'nombre_con_sufijo': [ch for ch in ch_names_total_str]
})

# Mostramos las primeras filas
print(df_canales.head())


# Añadimos la columna 'canal_efectivo': si está en el set, lo ponemos, si no, NaN
df_canales[f"canal_efectivo"] = df_canales['nombre_con_sufijo'].apply(
    lambda ch: ch if ch in canales_efectivos else np.nan
)

indices_efectivos = df_canales.loc[df_canales[f'canal_efectivo'].notna(), 'indice'].to_list()

# Paso 2: Recortar la matriz de adyacencia original
from scipy.sparse import csr_matrix

# adjacency_total es sparse, así que podemos hacer slicing con arrays de índices
adjacency_reducida = adjacency[indices_efectivos, :][:, indices_efectivos]

# Confirmamos la forma
print(f"✅ adjacency_reducida creada con forma: {adjacency_reducida.shape}")

59
Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 59
(59, 59)
   indice nombre_base nombre_con_sufijo
0       0         Fp1               Fp1
1       1         Fpz               Fpz
2       2         Fp2               Fp2
3       3         AF7               AF7
4       4         AF3               AF3
✅ adjacency_reducida creada con forma: (59, 59)


In [8]:
# Nombre completo del archivo
csv_path = os.path.join(channels_structure_path, f"channels_eeg.csv")

# Guardar el DataFrame
df_canales.to_csv(csv_path, index=False)

print(f"✅ Archivo guardado como: {csv_path}")

✅ Archivo guardado como: g:\PROYECTO_SELF\channels_structure\channels_eeg.csv


In [9]:
####lectura de la matriz de adyacencia (tiene que ir despue por fuerza)

import pickle
with open(channels_structure_path / f"adjacency_reduced.pkl", "wb") as f:
    pickle.dump(adjacency_reducida, f)
